# W12-D6 实验：五层图是 DAG 不是楼梯 · MallSenseAI 坐标的可执行验证

配套阅读材料：`第12周-Day6-Vision五层图与场景矩阵交付.md`（图和结论在那边，这里是**可运行证据**）。

三个实验，对应 md 的三个核心论断：
1. **层约束验证**：把五层模型建成代码里的有向图，验证①每条能力依赖边只指向更低层（DAG 性质成立）②状态型路径合法跳过 L2（楼梯假设被证伪）
2. **层覆盖率与场景层缺口**：9 个商业场景的"所需层 vs MallSenseAI 已有层"矩阵计算，验证"当期 ROI 最高行 = 层缺口为 0 的行"
3. **为什么计数是 L2 不是 L1**：模拟行人过线计数——纯帧级检测（无 ID）的过计数倍数 = 平均在视野帧数；静态误报（广告画假人）在无跟踪时是永久性偏差。这就是"L3 退化形态的能力边界来自 L2 缺失"的数学形态。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体就绪:", font_name)

## 实验 1：五层依赖图 —— 验证"每层只依赖下层"与"状态型路径跳 L2"

把五层模型写成代码：节点 = 能力（带 `layer` 属性与 MallSenseAI 占用状态），边 = 凭证依赖（上层的账要引用下层的凭证）。

**检验 1（DAG 层约束）**：所有依赖边必须从高层指向低层 —— 验证"每层只依赖下一层"在图上的严格形式。
**检验 2（楼梯假设证伪）**：楼梯模型断言"L4 的账必须引用 L2 的轨迹凭证"。检查 `工单闭环(L4')` 的传递依赖闭包里**有没有 L2 节点** —— 没有，则状态型路径合法跳层，五层模型是 DAG 不是楼梯。

In [ ]:
# ---- 能力节点：(名称, 层号, MallSenseAI 是否已占用) ----
# 层号: L1=1 ... L5=5；3.5≈L3' 退化统计, 4.5≈L4' 雏形（用分数避免与标准层混淆）
nodes = {
    "帧级检测(L1)":        (1, True),   # yolo11n / D-Fire / YOLO-World / 图像比对
    "开放词表零样本(L1)":   (1, True),
    "跟踪Tracking(L2)":    (2, False),
    "占用统计(L3')":       (3.5, True), # duration/area 规则，无跟踪的退化形态
    "密度计数(L3)":        (3, False),
    "客流KPI(L3)":         (3, False),
    "工单闭环(L4')":       (4.5, True), # 告警→工单→resolved
    "运营报表(L4)":        (4, False),
    "自动日报(L5)":        (5, False),
}
# 依赖边：上层能力 -> 它记账必须引用的下层凭证
edges = [
    ("跟踪Tracking(L2)", "帧级检测(L1)"),
    ("占用统计(L3')", "帧级检测(L1)"),        # 状态型：直接建在 L1 上（跳过 L2）
    ("密度计数(L3)", "跟踪Tracking(L2)"),      # 事件/分析型：必须引用轨迹凭证
    ("客流KPI(L3)", "跟踪Tracking(L2)"),
    ("工单闭环(L4')", "占用统计(L3')"),
    ("运营报表(L4)", "客流KPI(L3)"),
    ("自动日报(L5)", "工单闭环(L4')"),         # L5 可直接建在状态型闭环上（Day5 判定④）
]

# ---- 检验 1：层约束（所有边从高层指向低层）----
violations = [(a, b) for a, b in edges if nodes[a][0] <= nodes[b][0]]
print("检验1 层约束（边只能从高层指向低层）:", "PASS ✓" if not violations else f"FAIL {violations}")

# ---- 检验 2：楼梯假设证伪 ----
def closure(start):
    seen, frontier = set(), [start]
    while frontier:
        cur = frontier.pop()
        for a, b in edges:
            if a == cur and b not in seen:
                seen.add(b); frontier.append(b)
    return seen

dep_ticket = closure("工单闭环(L4')")          # 状态型 L4 的传递依赖
dep_report = closure("运营报表(L4)")            # 分析型 L4 的传递依赖
l2_in_ticket = any(nodes[n][0] == 2 for n in dep_ticket)
print(f"检验2a 工单闭环(L4')依赖闭包: {sorted(dep_ticket)}")
print(f"       其中 L2 节点: {'有 → 楼梯成立' if l2_in_ticket else '无 → 合法跳层，楼梯假设被证伪 ✓'}")
print(f"检验2b 运营报表(L4)依赖闭包含 L2: {any(nodes[n][0]==2 for n in dep_report)}（分析型确实绕不开 L2）")

In [ ]:
# ---- 可视化：DAG 与两条价值路径 ----
pos = {
    "帧级检测(L1)":        (0.5, 0.5),
    "开放词表零样本(L1)":   (1.5, 0.5),
    "跟踪Tracking(L2)":    (2.6, 1.5),
    "占用统计(L3')":       (0.9, 2.4),
    "密度计数(L3)":        (2.2, 3.0),
    "客流KPI(L3)":         (3.2, 2.6),
    "工单闭环(L4')":       (1.2, 3.8),
    "运营报表(L4)":        (3.0, 4.2),
    "自动日报(L5)":        (1.6, 5.0),
}
状态型 = {"帧级检测(L1)","占用统计(L3')","工单闭环(L4')","自动日报(L5)"}

fig, ax = plt.subplots(figsize=(10.5, 6.2))
for a, b in edges:
    on_state = a in 状态型 and b in 状态型
    ax.annotate("", xy=pos[b], xytext=pos[a],
                arrowprops=dict(arrowstyle="-|>", lw=2.6 if on_state else 1.2,
                                color="#dd8452" if on_state else "#999999",
                                shrinkA=26, shrinkB=26))
for name, (layer, owned) in nodes.items():
    x, y = pos[name]
    if owned:   fc, ec, lw = "#4c72b0", "#2f4870", 2.2   # 已占用：实心蓝
    elif name in 状态型: fc, ec, lw = "#ffffff", "#4c72b0", 1.4  # 状态路径未占：白底
    else:       fc, ec, lw = "#ffffff", "#aaaaaa", 1.0   # 其他未占：灰
    ax.scatter([x],[y], s=2600, c=fc, edgecolors=ec, linewidths=lw, zorder=3)
    label = name.replace("(", "\n(")          # 两行排版，避免长名截断
    ax.text(x, y, label, ha="center", va="center", fontsize=8.4, zorder=4,
            color="white" if owned else "#333333")
# 图层带标注
for y0, y1, lab in [(0,1,"L1"),(1,2.2,"L2"),(2.2,3.4,"L3/L3'"),(3.4,4.6,"L4/L4'"),(4.6,5.4,"L5")]:
    ax.axhspan(y0, y1, color="#f5f5f5" if int(lab[1])%2==0 else "none", zorder=0)
    ax.text(4.05, (y0+y1)/2, lab, fontsize=11, color="#888888", va="center")
ax.set_title("Vision Capability 依赖图：橙色 = 状态型路径（跳过 L2）·蓝实心 = MallSenseAI 已占用", fontsize=11)
ax.set_xlim(0, 4.4); ax.set_ylim(0, 5.5); ax.axis("off")
plt.tight_layout(); plt.savefig("d6_layer_dag.png", dpi=130); plt.show()
print("图已存 d6_layer_dag.png")

## 实验 2：层覆盖率 × 场景层缺口 —— "当期 ROI 行 = 层缺口 0 行"

数据来自 md §6/§7 的代码事实（已占用格）与 9 场景需求。计算两件事：
1. 每层覆盖率（L1 ~70%、L2 0%、L3 15%、L4 20%、L5 0% 的出处）
2. 每个场景 `缺口 = 所需层 − 已有层`，验证 Day5/Day6 的结论：**前 3 行（已部署）层缺口恰为空集**

In [ ]:
# ---- MallSenseAI 层资产（来自 md §6.1 的代码事实）----
layer_assets = {
    "L1":  {"封闭词表","领域微调","开放词表","基线比对"},          # 4/6 格（OCR、分割VLM 未占）
    "L2":  set(),                                                  # OpenSpec 29 spec 无 video/stream/tracking
    "L3":  {"占用面积","停留时长"},                                # 退化形态 2/13 常见统计量
    "L4":  {"告警工单闭环","Dashboard统计"},                        # 雏形 2/10
    "L5":  set(),
}
grid_size = {"L1":6, "L2":8, "L3":13, "L4":10, "L5":8}  # 五层图各层格子总数（md §6.1 清点）

print("=== 各层覆盖率 ===")
for L in ["L1","L2","L3","L4","L5"]:
    n, d = len(layer_assets[L]), grid_size[L]
    bar = "█"*int(round(n/d*30))
    print(f"{L}: {n}/{d}  {bar} {n/d:.0%}")

# ---- 场景层缺口 = 所需层 − 已有层 ----
scenes = [  # (场景, Day5 ROI 排序, 所需层)
    ("消防通道占用", 1, {"L1","L3","L4"}),
    ("地面脏污",     2, {"L1","L4"}),
    ("火灾烟雾",     3, {"L1","L4"}),
    ("违停/占道/满溢",4, {"L1","L3"}),
    ("L5自动日报",   5, {"L1","L3","L4","L5"}),
    ("跌倒检测",     6, {"L1","L2","L4"}),
    ("夜间入侵",     7, {"L1","L2","L4"}),
    ("排队分析",     8, {"L1","L2","L3"}),
    ("客流/热区",    9, {"L1","L2","L3","L4"}),
]
print("\n=== 场景层缺口（所需 − 已有，L2 资产为空集）===")
ready = []
for name, rank, need in scenes:
    have = {L for L in need if layer_assets[L]}     # 该层有任一资产即算部分可建
    gap  = need - have
    ready.append((name, rank, len(need), len(gap)))
    print(f"{name:12s} ROI第{rank}  所需{len(need)}层 → 缺口 {sorted(gap) if gap else '∅ 已可建'}")

top3_gapless = all(g==0 for _,r,_,g in ready if r<=3)
print(f"\n验证: ROI 前 3 行层缺口是否全为 0 → {'是 ✓（当期 ROI 行 = 层缺口 0 行）' if top3_gapless else '否'}")

In [ ]:
# ---- 可视化：场景 × 层 覆盖热力图 ----
order = [s[0] for s in scenes][::-1]
M = np.zeros((len(scenes), 5))
for i, (name, rank, need) in enumerate([(s[0], s[1], s[2]) for s in scenes][::-1]):
    for j, L in enumerate(["L1","L2","L3","L4","L5"]):
        if L in need:
            M[i, j] = 2 if layer_assets[L] else 1     # 2=所需且已有(绿) 1=所需但缺(红)

from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#f0f0f0", "#d68484", "#8fce8f"])
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.imshow(M, cmap=cmap, vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(5), ["L1 检测","L2 跟踪","L3 场景统计","L4 商业智能","L5 Agent"])
ax.set_yticks(range(len(order)), [f"{n}" for n in order])
for i in range(len(order)):
    for j in range(5):
        if M[i, j]: ax.text(j, i, "缺" if M[i,j]==1 else "✓", ha="center", va="center",
                            fontsize=10, color="white" if M[i,j]==1 else "#1a4d1a", fontweight="bold")
ax.set_title("Business Scene Matrix 层覆盖视图：绿=所需且已有 · 红=层缺口 · 灰=不需要")
plt.tight_layout(); plt.savefig("d6_scene_coverage.png", dpi=130); plt.show()
print("图已存 d6_scene_coverage.png —— 注意 L2 列全红（单点依赖层），前3行无红格")

## 实验 3：为什么计数是 L2 不是 L1 —— 过线计数的过计数模拟

场景：N 个行人以不同速度穿过画面中的计数线，每人在视野内停留 `residence_frames` 帧（速度慢的人停留久）。

- **L1 纯帧级方案**：每帧把"检测到的人"累加 → 计数 = Σ(每人在视野内的帧数) ≈ 真值 × 平均停留帧数。**慢的人被重复计的次数更多**（老人/停留者被系统性高估——恰好是扶梯口最需要准的人群）
- **L2 跟踪方案**：贪心最近邻把逐帧检测关联成轨迹 ID，只对"ID 首次过线"计数 → ≈ 真值

再叠加 md §9 思考题的静态误报：一个广告画上的"假人"（永不离场）。L1 方案下它**每帧都贡献计数**（永久性偏差，随帧率线性放大）；跟踪方案下它只是一个永不过线的静态 ID，过线计数不受影响。

In [ ]:
rng = np.random.default_rng(7)

N_PEOPLE, N_FRAMES = 40, 300
speeds = rng.uniform(0.004, 0.03, N_PEOPLE)     # 水平速度：有人快有人慢
entry_x = -rng.uniform(0, 0.8, N_PEOPLE)        # 进入画面时间错开
lanes   = rng.uniform(0.05, 0.95, N_PEOPLE)     # 每人一条随机泳道（2D，避免一维堆叠的身份歧义）
persons = []
for pid in range(N_PEOPLE):
    x = entry_x[pid]; traj = []
    for f in range(N_FRAMES):
        if 0.0 <= x <= 1.0: traj.append((f, x, lanes[pid]))   # 在视野内才被检测到
        x += speeds[pid]
    persons.append(traj)

residence = np.array([len(t) for t in persons])   # 每人在视野内的帧数 = 被L1重复计的次数
print(f"真值: {N_PEOPLE} 人过线 | 平均在视野帧数: {residence.mean():.0f} 帧")

# ---- L1 纯帧级：每帧把检测数累加（无 ID 概念）----
l1_count = int(residence.sum())

# ---- L2 简易跟踪：贪心最近邻把逐帧检测关联成轨迹 ID，只对每个 ID 首次过线计数 ----
LINE, MAX_STEP = 0.5, 0.04
active, crossed, next_tid = {}, set(), 0
for f in range(N_FRAMES):
    dets = [(pid, x, y) for pid, traj in enumerate(persons) for ff, x, y in traj if ff == f]
    pairs = []                                            # (距离, 检测idx, 轨迹id) 一对一贪心
    for i, (pid, x, y) in enumerate(dets):
        for tid, (tx, ty, lf) in active.items():
            if f - lf <= 1:                               # 只与上一帧更新过的轨迹匹配
                d = float(np.hypot(tx - x, ty - y))
                if d <= MAX_STEP: pairs.append((d, i, tid))
    pairs.sort()                                          # 最有把握的配对先锁定
    used_i, used_t, assign = set(), set(), {}
    for d, i, tid in pairs:
        if i in used_i or tid in used_t: continue
        used_i.add(i); used_t.add(tid); assign[i] = tid
    new_active = {}
    for i, (pid, x, y) in enumerate(dets):
        tid = assign.get(i)
        if tid is None: tid = f"t{next_tid}"; next_tid += 1
        new_active[tid] = (x, y, f)
        if x >= LINE and tid not in crossed: crossed.add(tid)
    active = new_active                                    # 两帧未更新的轨迹自然消亡
l2_count = len(crossed)

# ---- 静态误报：广告画上的"假人"（永在画面 x=0.3 的泳道外，永不过线）----
FP_FRAMES = N_FRAMES
l1_with_fp = l1_count + FP_FRAMES                          # L1：在场每帧都计
l2_with_fp = l2_count                                      # 跟踪：静态 ID 不过线，不计

print(f"\nL1 纯帧级计数: {l1_count}  → 过计数倍数 = {l1_count/N_PEOPLE:.0f}× （≈平均停留帧数 {residence.mean():.0f}）")
print(f"L2 跟踪计数:   {l2_count}  → 误差 {abs(l2_count-N_PEOPLE)/N_PEOPLE:.0%}（贪心最近邻的少量 ID 切换，量级正确）")
print(f"\n加一个静态假人（不过线，永在视野）:")
print(f"  L1 计数: {l1_count} → {l1_with_fp}（+{FP_FRAMES}，偏差随帧率/时长线性放大）")
print(f"  L2 计数: {l2_count} → {l2_with_fp}（+0，静态 ID 永不过线）")

In [ ]:
# ---- 可视化：慢行者被系统性高估 + 计数对比 ----
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

ax = axes[0]
sc = ax.scatter(speeds, residence, c=residence, cmap="YlOrRd", s=42, edgecolors="gray", lw=0.4)
slow, fast = np.percentile(speeds, [33, 67])
ax.axvspan(fast, speeds.max(), color="#e8f0e8")
ax.annotate(f"慢行者（后1/3）\n平均被重复计 {residence[speeds<=slow].mean():.0f} 次/人",
            xy=(0.008, residence[speeds<=slow].mean()), xytext=(0.012, 260),
            arrowprops=dict(arrowstyle="->", color="#c44e52"),
            fontsize=9, color="#8b2f2f")
ax.annotate(f"快行者（前1/3）\n平均 {residence[speeds>=fast].mean():.0f} 次/人",
            xy=(0.026, residence[speeds>=fast].mean()), xytext=(0.016, 60),
            arrowprops=dict(arrowstyle="->", color="#4c72b0"),
            fontsize=9, color="#2f4870")
ax.set_xlabel("行走速度（画面宽度/帧）"); ax.set_ylabel("在视野帧数 = L1 重复计数次数")
ax.set_title("L1 无 ID 计数：慢行者被系统性高估（扶梯口最需准的人群）")

ax = axes[1]
labels = ["真值", "L1 纯帧级", "L2 跟踪", "L1+静态假人", "L2+静态假人"]
vals = [N_PEOPLE, l1_count, l2_count, l1_with_fp, l2_with_fp]
colors = ["#55a868", "#c44e52", "#4c72b0", "#9d3336", "#33507c"]
bars = ax.bar(labels, vals, color=colors)
ax.axhline(N_PEOPLE, ls="--", c="gray", lw=1)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+40, f"{v}", ha="center", fontsize=9)
ax.set_ylabel("过线计数"); ax.set_yscale("log")
ax.set_title(f"过线计数对比（N={N_PEOPLE}，log 轴）：L1 偏差 ~{l1_count//N_PEOPLE}×")
ax.tick_params(axis="x", labelrotation=15)
plt.tight_layout(); plt.savefig("d6_tracking_vs_detection.png", dpi=130); plt.show()
print("图已存 d6_tracking_vs_detection.png")

## 结论回填（对应 md 的三个论断）

| 实验 | 验证了什么 | md 位置 |
|---|---|---|
| 1 层约束 | 所有依赖边高层→低层（DAG 成立）；工单闭环(L4')的依赖闭包**不含 L2** → 状态型路径合法跳层，楼梯假设证伪 | §6.3 |
| 2 层覆盖 | L2 列全红 = 单点依赖层；ROI 前 3 行（消防/脏污/火灾）层缺口恰为 ∅ → "当期 ROI 行 = 层缺口 0 行" | §7 |
| 3 计数 | 无 ID 过线计数 ≈ 真值 × 平均停留帧数（帧率越高越失真，慢行者被系统性高估）；静态误报在 L1 下是随帧率线性放大的永久偏差，在跟踪下不过线不计 → **L3 退化形态的边界来自 L2 缺失** | §9 思考题 |

一句话：五层图的"层"是**凭证依赖的层**，不是施工顺序的层——MallSenseAI 的窄（L1 四格）和深（闭环到 L4'）是同一个架构决策的两面。